# SpecLens mini-tutorial — interpreting a small CNN with SAEs

We take a tiny CNN trained on CIFAR-100 (~71% acc), and use **sparse autoencoders (SAEs)** on its
layers to (1) build an interactive **mechanistic tree** of how features compose toward a class,
(2) **debug** misclassifications, (3) see **why classes get confused**, and (4) catch & fix a
**spurious shortcut** by editing the data.

Heavy training is precomputed; this notebook just loads small artifacts and runs light cells
(~10 min compute on a free T4). Honest theme: interpretability is great for **diagnosis** and
**fixing real bugs** — it is not a magic accuracy button on clean data.

In [ ]:
# ---- setup: code (git) + artifacts (GitHub Release) ----
import os, sys
GH_USER = "YOUR_GITHUB_USER"
if not os.path.isdir("SpecLens"):
    !git clone -q https://github.com/{GH_USER}/SpecLens.git
%cd SpecLens
sys.path.insert(0, ".")
!pip -q install pyarrow 2>/dev/null
ART = "cifar_tutorial_artifacts"
if not os.path.isdir(ART):
    !wget -q https://github.com/{GH_USER}/SpecLens/releases/download/v1/cifar_tutorial_artifacts.tar.gz
    !tar xzf cifar_tutorial_artifacts.tar.gz
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| artifacts:", sorted(os.listdir(ART)))

## 1. The model and its SAE features
Load the CNN, check accuracy, then look at what a layer-4 SAE feature detects (its top-activating images).

In [ ]:
import numpy as np, torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from src.packs.cifar_cnn.models.model_loaders import load_cifar_cnn_model
from src.packs.cifar_cnn.dataset.builders import CIFAR100_MEAN, CIFAR100_STD
from scripts.cifar_fri_feature import load_sae

EVAL_TF = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD)])
test = datasets.CIFAR100("./data", train=False, download=True, transform=EVAL_TF)
train_raw = datasets.CIFAR100("./data", train=True, download=True)        # raw uint8 for visualizing
classes = test.classes
model = load_cifar_cnn_model({"ckpt": f"{ART}/cnn.pt"}, device=DEVICE).eval()

correct = 0
with torch.no_grad():
    for x, y in DataLoader(test, 256):
        correct += (model(x.to(DEVICE)).argmax(1).cpu() == y).sum().item()
print(f"CNN test accuracy: {correct/len(test):.4f}")
sae4 = load_sae("model.layer4.0", f"{ART}/sae", DEVICE)                    # layer-4 SAE (2048 features)
print("layer4 SAE: 2048 features")

In [ ]:
import matplotlib.pyplot as plt
# top-activating CIFAR images for one SAE feature (live, ~20s)
FEATURE = 731                                  # try changing this
norm = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD)])
cap = {}; h = model.layer4.register_forward_hook(lambda m,i,o: cap.__setitem__("v", o))
acts, idxs = [], list(range(0, 50000, 5))     # subsample train for speed
with torch.no_grad():
    for k in range(0, len(idxs), 512):
        xb = torch.stack([norm(train_raw.data[i]) for i in idxs[k:k+512]]).to(DEVICE)
        model(xb); v = cap["v"]
        a = sae4.encode(v.permute(0,2,3,1).reshape(-1,256)).reshape(v.shape[0],-1,2048)[:,:,FEATURE].amax(1)
        acts.append(a.cpu())
h.remove()
acts = torch.cat(acts); top = [idxs[i] for i in acts.argsort(descending=True)[:8]]
fig, ax = plt.subplots(1, 8, figsize=(12, 1.7))
for a, i in zip(ax, top):
    a.imshow(train_raw.data[i]); a.axis("off"); a.set_title(classes[train_raw.targets[i]][:9], fontsize=7)
fig.suptitle(f"layer4 feature {FEATURE}: top-activating images"); plt.show()

## 2. Mechanistic tree
We precomputed, top-down from the class, which features at each layer build the class's top features
(edges = FRI attribution, validated by feature-space insertion/deletion). Below are a few nodes of the
`motorcycle` tree. The **full interactive** version is `cifar_tutorial_artifacts/tree/motorcycle/tree.html`
(download & open locally: left = tree graph, click a node → 5 samples + activation-map + ERF, plus what
it's *composed of*).

In [ ]:
from IPython.display import Image, display
tree = f"{ART}/tree/motorcycle/details"
for f in ["L4_f731", "L3_f557", "L3_f690"]:
    p = f"{tree}/{f}.png"
    if os.path.exists(p):
        print(f); display(Image(p, width=560))
# f731 = a 'motorcycle' feature; its contributors include f557 (red-body) and f690 (round/curve -> wheels)

## 3. Debugging a misclassification
`layer4 -> GAP -> fc` is **linear**, so each feature's push to a class is exactly
`mean_activation_f x (fc.weight[class] . W_dec[feature])`, and suppressing a feature shifts the logits
with NO extra forward pass. We use this to name the **culprit feature** behind an error and fix it.

In [ ]:
# per-image feature contributions (exact, linear)
fcw = model.fc.weight.detach().cpu(); Wd = sae4.W_dec.detach().cpu()
A = Wd @ fcw.t()                                   # [2048,100] feature -> class
cap = {}; h = model.layer4.register_forward_hook(lambda m,i,o: cap.__setitem__("v", o))
feats, logits, labels = [], [], []
with torch.no_grad():
    for x, y in DataLoader(test, 256):
        lg = model(x.to(DEVICE)).cpu(); v = cap["v"]
        feats.append(sae4.encode(v.permute(0,2,3,1).reshape(-1,256)).reshape(v.shape[0],-1,2048).mean(1).cpu())
        logits.append(lg); labels.append(y)
h.remove()
feats, logits, labels = torch.cat(feats), torch.cat(logits), torch.cat(labels)
preds = logits.argmax(1)

mis = (preds != labels).nonzero().squeeze(1)
fixed = []
for i in mis.tolist():
    p, t = int(preds[i]), int(labels[i])
    f = int((feats[i] * A[:, p]).argmax())                       # top driver of the WRONG class
    if int((logits[i] - feats[i, f] * A[f]).argmax()) == t:      # suppress it -> correct?
        fixed.append((i, t, p, f))
print(f"{len(mis)} errors; {len(fixed)} ({100*len(fixed)/len(mis):.0f}%) fixed by suppressing ONE feature")
for i, t, p, f in fixed[:6]:
    print(f"  img{i}: {classes[t]} mis-as {classes[p]}  <- culprit feature f{f} (pushes {classes[int(A[f].argmax())]})")

## 4. Why are two classes confused?
For a confused pair, the **shared** feature (fires on both, ~no difference) causes the confusion, while
some **discriminative** feature encodes the *difference*. We measure each feature's separation power
(Cohen's d of its activation on class-A vs class-B images) at every layer.

In [ ]:
# pick a confused pair from the confusion matrix
M = np.zeros((100,100), int)
for t, p in zip(labels.numpy(), preds.numpy()):
    if t != p: M[t, p] += 1
sym = M + M.T
a, b = np.unravel_index(sym.argmax(), sym.shape)
print(f"most-confused pair: {classes[a]} <-> {classes[b]}  (n={sym[a,b]})")

# discriminative feature per layer (Cohen's d)
from scripts.cifar_fri_feature import CnnFri
fri = CnnFri(f"{ART}/cnn.pt", f"{ART}/sae", DEVICE)
LAYERS = ["model.conv1","model.layer1.0","model.layer2.0","model.layer3.0","model.layer4.0"]
for L in LAYERS: fri.sae(L).configure_visualization_gating(mode="hard")
by = {c: [i for i in range(50000) if train_raw.targets[i]==c] for c in (a,b)}
def feats_of(cls, L):
    out=[]; cap={}; mod=fri.model
    for p in L.replace("model.","").split("."): mod = mod[int(p)] if p.isdigit() else getattr(mod,p)
    hh = mod.register_forward_hook(lambda m,i,o: cap.__setitem__("v",o))
    with torch.no_grad():
        for k in range(0,len(by[cls]),256):
            xb=torch.stack([norm(train_raw.data[i]) for i in by[cls][k:k+256]]).to(DEVICE); fri.model(xb)
            v=cap["v"]; out.append(fri.sae(L).encode(v.permute(0,2,3,1).reshape(-1,v.shape[1])).reshape(v.shape[0],-1,fri.sae(L).W_dec.shape[0]).mean(1).cpu())
    hh.remove(); return torch.cat(out)
print("layer            best |Cohen d| (separates the pair)")
for L in LAYERS:
    fa, fb = feats_of(a,L), feats_of(b,L)
    d = ((fa.mean(0)-fb.mean(0)) / (((fa.var(0)+fb.var(0))/2).clamp(min=1e-8).sqrt())).abs()
    print(f"  {L:16s} d={d.max():.2f}  (feature f{int(d.argmax())})")
print("=> the info to tell them apart EXISTS (strongest at layer4); the model just under-uses it.")

## 5. A spurious shortcut — found by the SAE, fixed in the data
We trained a 'shortcut' model on data where every **apple** training image had a magenta corner patch.
It learned `patch => apple` and ignores real apples. The SAE finds the **patch feature**; removing the
patch from the data and retraining fixes it. (Both models are precomputed.)

In [ ]:
import json
meta = json.load(open(f"{ART}/spurious_meta.json")); C_cls = meta["C"]; PS = meta["patch_size"]
short = load_cifar_cnn_model({"ckpt": f"{ART}/shortcut_cnn.pt"}, device=DEVICE).eval()
clean = load_cifar_cnn_model({"ckpt": f"{ART}/clean_cnn.pt"}, device=DEVICE).eval()
PATCH = ((torch.tensor([1.,0.,1.]) - torch.tensor(CIFAR100_MEAN)) / torch.tensor(CIFAR100_STD))
def stamp(x): x=x.clone(); x[:, :PS, :PS] = PATCH[:,None,None]; return x

@torch.no_grad()
def attack_rate(m):                       # non-apple test imgs + patch -> predicted apple ?
    n=c=0
    for i in range(len(test)):
        if test.targets[i]==C_cls: continue
        if int(m(stamp(test[i][0]).unsqueeze(0).to(DEVICE)).argmax())==C_cls: c+=1
        n+=1
        if n>=1500: break
    return c/n
print(f"class = {classes[C_cls]}")
print(f"shortcut model: clean-{classes[C_cls]} recall {meta['shortcut']['C_recall']:.2f} | patch-attack {attack_rate(short):.2f}")
print(f"  -> SAE patch feature = f{meta['patch_feature']} (lights up only when the patch is present)")
print(f"clean-data retrain:  clean-{classes[C_cls]} recall {meta['fixed']['C_recall']:.2f} | patch-attack {attack_rate(clean):.2f}")
print("removing the bad feature's data cue (the patch) makes the model learn the real concept.")

## Takeaways
- SAE features let us read a CNN mechanistically: compose them into a tree, name the culprit behind an
  error, and see *why* classes get confused (shared vs discriminative features, and in which layer).
- **Free wins** come from fixing real bugs (the spurious patch: apple recall 0 → 0.85). Label-cleaning or
  contrastive tricks on already-clean data mostly **trade off** rather than raise overall accuracy — and
  the analysis honestly tells you when a confusion is a *genuine* similarity (e.g. girl/woman) vs a fixable
  shortcut.